In [1]:
import pandas as pd
import readability
import os
import numpy as np
import random
import torch
from bert_score import BERTScorer
import re
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

In [2]:
from transformers import logging
logging.set_verbosity_error() # make sure only important transformers logging output is visible

In [3]:
INPUT_DIR = 'dataGeneration/processedData'
OUTPUT_DIR = 'outputScores'

In [4]:
# Only need to run once.
# nltk.download('all')

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
prompt_df = pd.read_csv('./promptDataPreparation/promptDataPreparation.csv')

In [7]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [8]:
dataset_readabilities = {}
# Get readability of full datasets. 
for folder in os.listdir('./getText/datasetsPrep'):
    if os.path.isdir(f'./getText/datasetsPrep/{folder}'):
        for file in os.listdir(f'./getText/datasetsPrep/{folder}'):
            if file.endswith('.csv'):
                temp_df = pd.read_csv(f'./getText/datasetsPrep/{folder}/{file}')
                text_series = temp_df['text'].fillna('').astype(str).str.replace('.', '.\n', regex=False)
                long_text = '\n'.join(text_series)
                dataset_readabilities[file] = readability.getmeasures(long_text, lang='en')['readability grades']['SMOGIndex']

dataset_readabilities

{'yahoo.csv': 8.457860209518776,
 'banking77.csv': 7.7227892552376485,
 'huffPostNews.csv': 9.800422735858234,
 'clinc150.csv': 7.115884635058898,
 'atis.csv': 9.558259963681241,
 'medicalAbstracts.csv': 14.371525966195462,
 'dementiaAudio.csv': 5.381402423935505,
 'syntheticCareHomeNurseNotes.csv': 10.337798278917836,
 'clinicalDialogueSummarizations.csv': 10.314872711433278,
 'simSUM.csv': 8.856790533456255}

In [9]:
average_scores_dict = []
# Make BERT scorer.
scorer = BERTScorer(model_type="bert-base-uncased")
for dataset in os.listdir(f"./{INPUT_DIR}"):
    if dataset.endswith('.csv'):
        temp_df = pd.read_csv(f"./{INPUT_DIR}/{dataset}")
        for topic_model in ['MATAVE', 'LDA']:

            topic_model_df = temp_df[temp_df['topic_model'] == topic_model]
            # --------- Readability (number linked to grade of reading level)s ---------
            references = []
            candidates = []
            meteors = []
            abs_sentiment_diffs = []
            abs_subjectivity_diffs = []
            # Make readability metric.
            reference_readability = dataset_readabilities[dataset]
            text_series = topic_model_df['report'].fillna('').astype(str).str.replace('.', '.\n', regex=False)
            long_text = '\n'.join(text_series)
            candidate_readability = readability.getmeasures(long_text, lang='en')['readability grades']['SMOGIndex']
            abs_readability_diff = (math.sqrt((reference_readability - candidate_readability) ** 2))
            for temp_prompt, temp_generation in zip(topic_model_df['example_text_in_prompt'], topic_model_df['report']):
                # Make candidates and references without punctuation for metrics (BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf).
                reference = re.sub(r'[^\w\s/]', '', temp_prompt)
                candidate = re.sub(r'[^\w\s/]', '', temp_generation)
                references.append(reference)
                candidates.append(candidate)
                # Make METEOR
                meteors.append(meteor([word_tokenize(candidate)], word_tokenize(reference)))
                # Make sentiment and subjectivity absolute differences.
                reference_blob = TextBlob(preprocess_text(temp_prompt)).sentences[0].sentiment
                candidate_blob = TextBlob(preprocess_text(temp_generation)).sentences[0].sentiment
                abs_sentiment_diffs.append(math.sqrt((reference_blob.polarity - candidate_blob.polarity) ** 2))
                abs_subjectivity_diffs.append(math.sqrt((reference_blob.subjectivity - candidate_blob.subjectivity) ** 2))
            # Make BERTScore
            _, _, F1 = scorer.score(candidates, references)

            average_scores_dict.append({
                'topic_model': topic_model, 
                'dataset': dataset, 
                'number_of_notes': len(topic_model_df),
                'abs_readability_diff': abs_readability_diff,
                # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
                'bertscore': float(F1.mean()),
                'meteor': sum(meteors) / len(meteors),
                'abs_sentiment_diff': sum(abs_sentiment_diffs) / len(abs_sentiment_diffs),
                'abs_subjectivity_diff': sum(abs_subjectivity_diffs) / len(abs_subjectivity_diffs)})


/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
# min-max normalize every score
resultant_df = pd.DataFrame(average_scores_dict)

In [24]:
scaled_df = resultant_df.copy()

metric_cols = [
    "abs_readability_diff",
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

# Min-max normalize safely.
for col in metric_cols:
    min_val = scaled_df[col].min()
    max_val = scaled_df[col].max()

    if max_val - min_val == 0:
        scaled_df[col] = 0.0
    else:
        scaled_df[col] = (scaled_df[col] - min_val) / (max_val - min_val)

# Invert the lower is better columns.
lower_is_better = [
    "abs_readability_diff",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

for col in lower_is_better:
    scaled_df[col] = 1 - scaled_df[col]

# Get overall scores.
scaled_df["overall_score"] = scaled_df[[
    "bertscore",
    "meteor",
    "abs_readability_diff",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]].mean(axis=1)

# Get final summary table.
summary = (
    scaled_df
    .groupby(["dataset", "topic_model"])["overall_score"]
    .mean()
    .reset_index()
    .sort_values(["dataset", "overall_score"], ascending=[True, False])
)

# Save outputs. 
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok=True)
scaled_df.to_csv(f'./{OUTPUT_DIR}/evaluated_scaled_results.csv', index=False)
resultant_df.to_csv(f'./{OUTPUT_DIR}/raw_evaluation_results.csv', index=False)
summary.to_csv(f'./{OUTPUT_DIR}/dataset_summary.csv', index=False)

In [25]:
print("------ Summary Performance ------")
summary

------ Summary Performance ------


,dataset,topic_model,overall_score
1,atis.csv,MATAVE,0.618681
0,atis.csv,LDA,0.458798
3,banking77.csv,MATAVE,0.531906
2,banking77.csv,LDA,0.404133
4,clinc150.csv,LDA,0.356829
5,clinc150.csv,MATAVE,0.330257
6,clinicalDialogueSummarizations.csv,LDA,0.463461
7,clinicalDialogueSummarizations.csv,MATAVE,0.426545
8,dementiaAudio.csv,LDA,0.793772
9,dementiaAudio.csv,MATAVE,0.569824


In [26]:
print("------ Scaled Scores ------")
scaled_df.sort_values(by='overall_score', ascending=False)

------ Scaled Scores ------


,topic_model,dataset,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,overall_score
4,MATAVE,medicalAbstracts.csv,348,0.303270,0.948111,1.000000,0.911304,0.985866,0.829710
17,LDA,simSUM.csv,636,0.168908,0.953225,0.900223,1.000000,1.000000,0.804471
7,LDA,dementiaAudio.csv,1394,0.671547,0.993696,0.771234,0.566380,0.966001,0.793772
16,MATAVE,simSUM.csv,756,0.000000,1.000000,0.913708,0.944380,0.908742,0.753366
18,MATAVE,atis.csv,824,1.000000,0.844411,0.291673,0.543211,0.414112,0.618681
5,LDA,medicalAbstracts.csv,2723,0.719491,0.479347,0.459613,0.656783,0.666756,0.596398
6,MATAVE,dementiaAudio.csv,1166,0.325822,0.809962,0.571434,0.362846,0.779054,0.569824
12,MATAVE,syntheticCareHomeNurseNotes.csv,1645,0.878353,0.678809,0.237894,0.584094,0.463028,0.568436
13,LDA,syntheticCareHomeNurseNotes.csv,625,0.749968,0.674409,0.399779,0.644648,0.347673,0.563295
2,MATAVE,banking77.csv,769,0.589636,0.693336,0.216036,0.705707,0.454816,0.531906


In [27]:
print("------ Raw Dataframe ------")
resultant_df

------ Raw Dataframe ------


,topic_model,dataset,number_of_notes,abs_readability_diff,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff
0,MATAVE,yahoo.csv,729,8.558779,0.453732,0.049500,0.221742,0.309920
1,LDA,yahoo.csv,1938,5.680751,0.456032,0.064882,0.202166,0.232044
2,MATAVE,banking77.csv,769,5.567340,0.503317,0.065824,0.121739,0.258111
3,LDA,banking77.csv,1032,3.413767,0.449180,0.053347,0.201398,0.285130
4,MATAVE,medicalAbstracts.csv,348,7.461355,0.551251,0.242330,0.090604,0.107558
5,LDA,medicalAbstracts.csv,2723,4.708488,0.463057,0.120664,0.129148,0.198026
6,MATAVE,dementiaAudio.csv,1166,7.312194,0.525259,0.145840,0.173662,0.166190
7,LDA,dementiaAudio.csv,1394,5.025586,0.559827,0.190824,0.142839,0.113190
8,MATAVE,huffPostNews.csv,490,4.654778,0.409861,0.032914,0.142408,0.387052
9,LDA,huffPostNews.csv,4522,4.971174,0.372872,0.017184,0.228612,0.299712


In [33]:
# Global model comparison. 
global_summary = (
    scaled_df
    .groupby("topic_model")[[
        "bertscore",
        "meteor",
        "abs_readability_diff",
        "abs_sentiment_diff",
        "abs_subjectivity_diff",
        "overall_score"
    ]]
    .mean()
    .sort_values("overall_score", ascending=False)
)

# Winning rate for each model.
pivot = scaled_df.pivot_table(
    index="dataset",
    columns="topic_model",
    values="overall_score"
)

pivot["MATAVE_better_performance"] = pivot["MATAVE"] > pivot["LDA"]

print("\n------ MATAVE Best Performance Rate ------")
print(pivot["MATAVE_better_performance"].mean())


------ MATAVE Best Performance Rate ------
0.5


In [34]:
pivot

topic_model,LDA,MATAVE,MATAVE_better_performance
dataset,,,
atis.csv,0.458798,0.618681,True
banking77.csv,0.404133,0.531906,True
clinc150.csv,0.356829,0.330257,False
clinicalDialogueSummarizations.csv,0.463461,0.426545,False
dementiaAudio.csv,0.793772,0.569824,False
huffPostNews.csv,0.197570,0.312661,True
medicalAbstracts.csv,0.596398,0.829710,True
simSUM.csv,0.804471,0.753366,False
syntheticCareHomeNurseNotes.csv,0.563295,0.568436,True


In [30]:
print("\n------ Global Model Performance ------n")
global_summary


------ Global Model Performance ------n


,bertscore,meteor,abs_readability_diff,abs_sentiment_diff,abs_subjectivity_diff,overall_score
topic_model,,,,,,
MATAVE,0.649131,0.365831,0.460241,0.562737,0.535563,0.514700
LDA,0.563135,0.352028,0.597464,0.452764,0.548748,0.502828
